# 03 — Haiku reference, independent judging and consolidation

Run locally with the new Python 3.12 environment. No tunnel is required. Each request has an independent context. All returned judge text is retained; parsing only adds audit columns. API errors and unknown submissions are distinct from poor model answers. Haiku uses provider-default temperature (the API does not support temperature=0) and disabled thinking. The 128,000 reference output limit and 8,192 judge output limit are actual recorded wire parameters. Estimated USD cost uses the 2026-10-24 pricing snapshot; token usage and receipts are retained.

## 1. Local environment and watchdog

Locate project files and verify the independent audible monitor.

**Recovery / safe rerun:** On a kernel restart, rerun 1–2, then the interrupted stage. Use notebook 00 section 5 if the watchdog stopped.

**Cost:** No paid API requests.



In [2]:
from pathlib import Path
import sys
candidates = [Path.cwd(), *Path.cwd().parents]
ROOT = next((p for p in candidates if (p/'streamlined/common.py').is_file()), None)
if ROOT is None:
    ROOT = next(p/'streamlined_experiment' for p in candidates if (p/'streamlined_experiment/streamlined/common.py').is_file())
sys.path.insert(0, str(ROOT))
from streamlined.common import *
print('Experiment:', ROOT)
print('Interpreter:', sys.executable)

from streamlined.notebook import watchdog_ready
watchdog_ready(ROOT)

Experiment: d:\reit4841_code\1-58-bit-llm-CAM-LDS\streamlined_experiment
Interpreter: d:\reit4841_code\1-58-bit-llm-CAM-LDS\streamlined_experiment\.venv-local\Scripts\python.exe


## 2. API connection

Read secret/Claude_API_Key.txt without displaying it. This metadata request verifies the exact model identifier without generating responses.

**Recovery / safe rerun:** Rerun 1–2 after an API-key change or kernel restart. The key is not written to notebook outputs.

**Cost:** No paid API requests.



In [3]:
from streamlined.api import client, APIRun, haiku_jobs, judge_jobs, COMMON_JUDGE_RUN, HAIKU_SELF_JUDGE_RUN
from streamlined.analysis import compile_tables, require_complete, consolidate_judgments, intervals
api = client(ROOT)
model_info = api.models.retrieve(POLICY['haiku_model'])
if model_info.id != POLICY['haiku_model']:
    raise ValueError('API returned a different model ID')
print('API access confirmed:',model_info.id)

API access confirmed: claude-haiku-5-5


## Dataset assembly for Section 3

Section 3 creates the lossless tables used by every later experiment in this notebook. Its source identity fields come from the 990 designated teacher rows prepared from `llm_responses.json`: unchanged prompt text, prompt ID and hash, scenario, variant, step, scenario-step group, train/validation/test split, and cleaned ground-truth ATT&CK technique IDs.

Candidate responses are joined from notebooks 01 and 02 without rewriting their text:

- Six untreated-HF models x 990 prompts = **5,940** expected rows.
- Six models x four SFT adapters x 140 validation prompts = **3,360** expected rows.
- Six models x four SFT adapters x 145 test prompts = **3,480** expected rows.
- `D_all_responses.csv` therefore contains **12,780 expected candidate records**, including explicit pending, error, and `context_unsupported` status rows.
- `E_technique_validation.csv`, `E_explanation_validation.csv`, and `E_joint_validation.csv` each contain six models x 140 validation prompts = **840 descriptive records**. They do not create separate paid judge runs.

The four SFT conditions are technique, explanation, joint, and full. Prompt hashes and source keys are checked before joining. Section 3 performs no inference and is safe to rebuild when additional candidate responses arrive.


## 3. Compile D and E before Haiku generation

D contains all expected raw/SFT rows, including pending/error/status rows. Three E files remain non-paid validation views for technique, explanation and joint across all six models. Every completed candidate receives the same common rubric. Source keys and prompt hashes are checked.

**Recovery / safe rerun:** Rerun after any additional small-model results arrive. This updates views, never launches generation.

**Cost:** No paid API requests.



In [4]:
compiled = compile_tables(ROOT)
print('D expected records:',len(compiled))
print('Tables:',ROOT/'results/compiled')

D expected records: 12780
Tables: d:\reit4841_code\1-58-bit-llm-CAM-LDS\streamlined_experiment\results\compiled


## Experiment A - Haiku reference/baseline generation (Sections 4-6)

Claude Haiku is queried once in an independent context for each of the same **990 unchanged CAM-LDS prompts**. The five-row pilot and main batch run share one resumable ledger, so pilot rows are not purchased twice. The result preserves raw Haiku text, prompt identity, scenario/variant/step/split metadata, ground truth, token usage, estimated cost, model identifier, API receipts, and request status.

This experiment supplies both (a) a standalone Haiku baseline and (b) the comparison response used only for the common rubric's similarity fields. It does not make the Haiku answer authoritative, modify teacher data, train a model, or select an SFT adapter.


## 4. Haiku five-row pilot

Paid independent requests. Print each response, token usage and estimated cost. These are the first five missing IDs from the same 990-row ledger; the main run reuses them.

**Recovery / safe rerun:** Rerun 1–2 then reconstruct haiku here. Completed responses are skipped. An unknown synchronous request is held to avoid a duplicate charge; restore a saved receipt or investigate the API receipt before explicit retry.

**Cost:** No paid API requests.

Up to five paid Messages API requests; no quality-dependent retry.

In [5]:
haiku = APIRun(ROOT,'haiku',haiku_jobs(ROOT),api)
haiku.restore_receipts()
print(haiku.pilot(size=5))

Paid pilot requests: 5
1 ```json
{
  "techniques": ["T1590.002", "T1596.001", "T1046", "T1018", "T1595.002", "T1590.001", "T1135", "T1046.001", "T1592", "T1482"],
  "confidence": "Almost Certain: Attack",
  "explanation": "A single host (192.42.1.174) issues a rapid burst of A-record queries for many distinct, plausibly named internal hostnames (nagios, mrtg, s2, mailadmin, dev2) within the same second, all returning NXDOMAIN. This systematic enumeration of subdomains and hostnames is characteristic of DNS reconnaissance/brute-forcing rather than normal user browsing."
}
```
Usage/cost: {'cache_creation': {'ephemeral_1h_input_tokens': 0, 'ephemeral_5m_input_tokens': 0}, 'cache_creation_input_tokens': 0, 'cache_read_input_tokens': 0, 'inference_geo': 'global', 'input_tokens': 885, 'output_tokens': 239, 'output_tokens_details': {'thinking_tokens': 0}, 'server_tool_use': None, 'service_tier': 'standard', 'speed': 'standard'} 0.000208
2 ```json
{
  "techniques": ["T1059.004", "T1059", "T10

## 5. Submit remaining Haiku references

Submit batches of 50. Persist intent before submission and remote batch ID immediately afterwards. Never rerun a completed prompt because of malformed or empty output.

**Recovery / safe rerun:** Rerun 1–2 and 5 after restart. Unknown submissions remain held; reconcile after remote completion. Use section 10 for requests with confirmed API failures.

**Cost:** Paid Batch API requests for missing rows.



In [6]:
haiku = APIRun(ROOT,'haiku',haiku_jobs(ROOT),api)
print(haiku.reconcile())
print(haiku.submit(batch_size=50))

{'total': 990, 'completed': 5, 'pending': 985}
Paid batch requests: 985
Submitted msgbatch_014broLmyKmrHJacNPMibN7H 50
Submitted msgbatch_01W9YaPNBUS43fKdt1KE1dwp 50
Submitted msgbatch_013Mx4WPQJSCjRodHCbAfRUJ 50
Submitted msgbatch_01XN76vDjSxrAsH3wUAVGgSt 50
Submitted msgbatch_018s8fXXfDUaA7w5zoUVZGiT 50
Submitted msgbatch_01Bc7TLVUkBeC9LMX961rEmR 50
Submitted msgbatch_01KLBtEXgCUEp4JLzCUFRReS 50
Submitted msgbatch_0178dGratpoBqXZLXEe7c9bP 50
Submitted msgbatch_014NkyhmzrYMgeeT1HLF6hU9 50
Submitted msgbatch_01KGLQZgr6pMFouyUJzB3pCS 50
Submitted msgbatch_017QygrMCKpPLDjftL9rrUZh 50
Submitted msgbatch_01SMqVha8Ptj2HUoErx8GA4x 50
Submitted msgbatch_01L5QRvZzLv9v9Vi3Kn93YST 50
Submitted msgbatch_0113it5xCGUQiCtHyaE3wcnM 50
Submitted msgbatch_01FWRyo31NcG2z1LnJxY7X1n 50
Submitted msgbatch_01EkYTvDyzj4w5PwUJYGXDVB 50
Submitted msgbatch_01DP4src9R4GqRGGngLLjKrc 50
Submitted msgbatch_01GtpmNaM3K8GhKQaNGwnZ3R 50
Submitted msgbatch_011fzt85LguTGaejGAUQ7FLU 50
Submitted msgbatch_01QA3adWGxF2KLjL

## 6. Poll and save Haiku responses

Polling saves each returned response locally. A poll window ending does not mean all batches have finished. No new paid work is submitted here.

**Recovery / safe rerun:** Rerun 1–2 then 6 until completed=990, or use 10 for confirmed failures. Rerun 5 to reconcile an unknown batch.

**Cost:** No paid API requests.



In [7]:
haiku = APIRun(ROOT,'haiku',haiku_jobs(ROOT),api)
print(haiku.poll(max_seconds=3600))

{'total': 990, 'completed': 990}
{'total': 990, 'completed': 990}


## Experiments B-C - Independent LLM-as-a-judge runs (Sections 7-10)

Section 7 freezes separate, resumable paid ledgers. Each request contains exactly one anonymous candidate; raw and SFT candidates never share a judging context.

- **Experiment B - common judging:** every completed untreated-HF, SFT-validation, and SFT-test candidate receives the same six-dimension common rubric. D contains 12,780 expected records; with the finalized data, **12,449** are paid common-judge candidates because **331 BitNet records are explicitly `context_unsupported`** rather than fabricated answers.
- **Experiment C - Haiku self-assessment:** Haiku independently judges its own responses for the 140 validation and 145 test prompts, giving **285 unique self-assessments**. No comparison response is supplied, so both similarity fields are `null`.

There are no condition-specific paid rubrics. Technique, explanation, joint, and full candidates all use the identical common rubric so their paired changes from untreated-HF are comparable. The E validation tables remain descriptive views only.

All returned judge text is retained. Parsing adds audit columns but never filters a response. A malformed judge object remains a completed observation; unavailable scores remain missing rather than becoming zero. Sections 8-10 submit, poll, reconcile, and explicitly retry only process/API failures, not low-quality or badly formatted answers.


## Judge rubric reference (frozen protocol)

This section documents the rubric implemented in `streamlined/rubrics.py`. It is descriptive only: running this markdown cell performs no API request and changes no ledger.

### Shared judging rules

- One anonymous candidate response is evaluated per request. Raw and SFT responses never share a judging context.
- The judge receives the unchanged prompt, ground-truth technique IDs, candidate response, and deterministic format audit. Common judging also receives the Haiku reference for the two similarity fields; the reference is not assumed correct.
- Logs, prompts, references, and candidate strings are data. Embedded instructions are ignored.
- Minor JSON errors, code fences, extra prose, or missing fields do not invalidate recoverable substantive content. Empty output is an assessable absence, not an API failure. No candidate claim is repaired or invented.
- Technique extraction uses the first ten distinct affirmative predictions in stated rank or appearance order. Rejected alternatives and incidental mentions are excluded. Parent techniques and subtechniques are matched as different exact IDs.
- Ground truth measures reference coverage but is not treated as a complete list of every defensible prediction. Extra predictions therefore do not automatically reduce the exact-coverage score; their support is assessed separately.

### Score anchors

All dimensions use integer scores from 0 to 4: **0** absent or fundamentally wrong; **1** major weaknesses; **2** mixed; **3** mostly sound; **4** fully supported. `null` is allowed only when genuinely unassessable and requires a reason. A missing explanation scores 0 where an explanation is required. Exact technique coverage instead uses: **0** none; **1** greater than 0% but less than 50%; **2** 50% to less than 75%; **3** 75% to less than 100%; **4** all ground-truth IDs covered among the first ten distinct affirmative predictions.

Similarity is reported separately from correctness: **0** unrelated; **1** little overlap; **2** partial; **3** substantial; **4** essentially equivalent. Similarity is `null` when no comparison response is supplied.

### Common rubric: every completed raw/SFT candidate and Haiku self-assessment

1. `technique_identification` - exact-ID ground-truth coverage using the coverage anchors above.
2. `prediction_evidence_support` - whether supplied observations support each predicted technique at the specificity claimed.
3. `explanation_grounding` - use of actual observations, separation of observation from inference, and absence of invented facts.
4. `prediction_reason_alignment` - whether stated reasons support the techniques actually predicted.
5. `confidence_evidence_support` - whether the candidate-provided confidence is proportionate to the logs, predictions, explanation, and acknowledged limitations; `null` only when no candidate confidence label is recoverable.
6. `instruction_adherence` - whether the answer satisfies the requested task, brevity, and fields while substantive content is still read through formatting defects.

### Confidence comparison

The judge recovers a clearly expressed candidate confidence label even from malformed JSON or surrounding prose, but never invents one from tone. It independently chooses a warranted label from the same seven CAM-LDS categories. `confidence_label_agreement` is scored 0-4: 4 exact; 3 adjacent with the same Attack/Normal direction; 2 moderate difference or Neutral involvement; 1 major difference or weak directional disagreement; 0 opposing high-confidence conclusions; `null` when no candidate label is recoverable.

### Returned and retained audit fields

The requested judge object contains `scores`, short evidence-linked `reasons`, `null_reasons`, `extracted_predictions`, `failure_labels`, `prediction_similarity`, `reasoning_similarity`, `candidate_confidence_label`, `judge_warranted_confidence_label`, `confidence_label_agreement`, `confidence_label_agreement_reason`, and `confidence_label_null_reason`. `failure_labels=[]` means no observed failure; `null` means unavailable. Raw judge text is always retained. Parsing adds columns but never rejects a response: malformed judge JSON remains a completed observation, unavailable scores remain missing rather than becoming zero, and response format diagnostics remain separate from rubric scores.

Candidate, teacher, Haiku-reference, and judge fields use explicit prefixes in compiled/final outputs: for example `candidate_response_raw`, `candidate_valid_json`, `teacher_response_raw`, `teacher_confidence_label`, `haiku_reference_response_raw`, `judge_response_raw`, and `judge_valid_json`. The judge-returned `candidate_confidence_label` is its recovery of the candidate's expressed label; `judge_warranted_confidence_label` is its independent evidential assessment. The teacher response is retained for audit but is not shown to the judge. Candidate-format failures are descriptive and never filter a completed response.

### Evaluation coverage

- The common rubric covers every completed untreated-HF, validation, and test candidate from all six models and all four adapters.
- Technique, explanation, joint, and full candidates all receive the same common rubric; the E validation files are non-paid descriptive views.
- Haiku self-assessment is a separate common-rubric run over Haiku responses for validation and test prompts. It has no comparison response, so similarity fields are `null`.


## 7. Join Haiku references and freeze judge jobs

Recompile D/E with corresponding Haiku responses. Require all expected candidate rows to be completed or explicitly context-unsupported before freezing requests. A single candidate is judged in each context; raw and SFT never share a judging request. Haiku self-assessment runs separately, once per validation/test prompt.

**Recovery / safe rerun:** After kernel restart rerun 1–2 and 7. If candidate outputs are missing, finish those model notebooks first. Never edit prompt/rubric settings inside an existing paid ledger.

**Cost:** No paid API requests.



In [8]:
compiled = compile_tables(ROOT)
require_complete(compiled)
runs = {COMMON_JUDGE_RUN:APIRun(ROOT,COMMON_JUDGE_RUN,judge_jobs(compiled),api)}
haiku = APIRun(ROOT,'haiku',haiku_jobs(ROOT),api)
self_rows = [r for r in haiku.store.rows.values() if r.get('split') in ('validation','test')]
runs[HAIKU_SELF_JUDGE_RUN] = APIRun(ROOT,HAIKU_SELF_JUDGE_RUN,judge_jobs(self_rows,self_assessment=True),api)
print({name:run.counts() for name,run in runs.items()})

{'common_uniform_v2': {'total': 12449, 'pending': 12449}, 'haiku_self_uniform_v2': {'total': 285, 'pending': 285}}


## 8. Submit uniform common and self-assessment judgments

All candidates use the same six dimensions. Confidence agreement and Haiku similarity are returned as additional comparable fields.

**Recovery / safe rerun:** Rerun 1–2, 7 and 8 after interruption. Completed/submitted jobs are skipped; unknown submissions stay held.

**Cost:** No paid API requests.

Paid batch judging. Counts are printed before each group is submitted.

In [9]:
for name, run in runs.items():
    print(name,run.reconcile())
    print(run.submit(batch_size=50))

common_uniform_v2 {'total': 12449, 'pending': 12449}
Paid batch requests: 12449
Submitted msgbatch_015FCFmAtfRyhKMCQP7ZA9mm 50
Submitted msgbatch_01P4kQV5ijnc96PaKUJjCKJG 50
Submitted msgbatch_01WjDQo8M3Xe1yaRNQRzvTPF 50
Submitted msgbatch_018GnjdLoFjsZAwVF9ppE6nf 50
Submitted msgbatch_01PSN61X9uSRGHKvPfEtFLyA 50
Submitted msgbatch_01QKCCnwpMNdg8thz2HLxfVR 50
Submitted msgbatch_01ESrvySMixJuDZfxza2yRci 50
Submitted msgbatch_01QgfdeGnY3MJGCYEzc9MkCi 50
Submitted msgbatch_01Xff82wriAtSAALpe5ZeMRT 50
Submitted msgbatch_01KUNULvDHkZCSaNKLyK6HAM 50
Submitted msgbatch_01CkxdBL4t7smZxmtRHWvCWF 50
Submitted msgbatch_012VmTnNrrCSxnzvfhc9Jv4x 50
Submitted msgbatch_01499w744rt4Qp2dSVtGzCEN 50
Submitted msgbatch_019kkAHWk9iGuTUkuehhsbkH 50
Submitted msgbatch_01UDzzALGqA7CmLsj6TzDQue 50
Submitted msgbatch_015WVc8hw7bjQksKeHssQqcw 50
Submitted msgbatch_01CdUDvrzTzNfcwAGQRGfYjQ 50
Submitted msgbatch_01Sf9sqCRimkbUjwAdtkMYpm 50
Submitted msgbatch_0126qPgrTiYpGm6Jb4xwefnz 50
Submitted msgbatch_017AhYqD

## 9. Poll judgments and consolidate

Malformed judge outputs remain completed observations. Missing extracted scores stay missing, never zero. Consolidation retains raw judge text, candidate text, usage and audit data.

**Recovery / safe rerun:** Rerun 1–2, 7 and 9 until no submitted jobs remain. Section 10 is only for confirmed API failures.

**Cost:** No paid API requests.



In [11]:
for name,run in runs.items():
    print(name,run.poll(max_seconds=3600))
judged = consolidate_judgments(ROOT)
print('Consolidated rows:',len(judged))

{'total': 12449, 'completed': 12449}
common_uniform_v2 {'total': 12449, 'completed': 12449}
{'total': 285, 'completed': 285}
haiku_self_uniform_v2 {'total': 285, 'completed': 285}
Consolidated rows: 12734


## 10. Resume missing or confirmed failed requests

This is an explicit retry of process/API failures. It never rejudges a received response for formatting or quality. Unknown synchronous requests require an audited resolution.

**Recovery / safe rerun:** After restart rerun 1–2, rebuild haiku (5/6) or runs (7), then 10. Repeat polling, not submission, for pending work.

**Cost:** No paid API requests.

Paid retries only for pending/rejected/confirmed failed requests.

In [ ]:
RETRY_TARGET = 'haiku'  # or COMMON_JUDGE_RUN, HAIKU_SELF_JUDGE_RUN
target = haiku if RETRY_TARGET=='haiku' else runs[RETRY_TARGET]
print(target.reconcile())
print(target.submit(batch_size=50,retry_failed=True))
print(target.poll(max_seconds=3600))

## 10a. Optional resolution of an unknown synchronous pilot

Use only after checking the saved receipt/API outcome. A lost synchronous response is not always recoverable. Retrying it may create a duplicate charge. Batch uncertainty must use reconciliation instead; this cell refuses batch records.

**Recovery / safe rerun:** Safe default does nothing. Reconstruct the target run first. A documented resolution enables section 10 retry or restores a receipt without another API call.

**Cost:** No paid API requests.



In [ ]:
RESOLVE_UNKNOWN = False
RECORD_ID = ''
RESOLUTION_REASON = ''
RECOVERED_RECEIPT = None  # Path to a recovered complete API message JSON, if available
ALLOW_POSSIBLE_DUPLICATE_CHARGE = False
if RESOLVE_UNKNOWN:
    target.resolve_unknown_pilot(RECORD_ID,RESOLUTION_REASON,
        receipt_path=RECOVERED_RECEIPT,allow_retry=ALLOW_POSSIBLE_DUPLICATE_CHARGE)

## Experiment D - Matched raw-versus-SFT analysis (Section 11)

The final analysis compares each model's untreated-HF response with the same model and prompt under each SFT adapter, separately for validation and test. Pairing requires identical prompt hashes. All candidates use the same common rubric. The analysis includes six quality scores, prediction similarity, reasoning similarity, and confidence-label agreement.

Uncertainty intervals use grouped bootstrap resampling at the scenario-step group level so related rows are not treated as independent. The output reports available matched pairs and preserves missing values. These comparisons are exploratory and do not select a winning adapter or feed any result back into training.


## 11. Final CSVs and grouped paired intervals

The consolidated CSV is the new charts input. Matched comparisons require identical prompt hashes. Bootstrap scenario-step groups; report available pairs per metric and keep nulls. Comparisons are exploratory, without multiplicity-adjusted claims.

**Recovery / safe rerun:** Safe to rerun after any recovered results. It does not submit requests or change source responses.

**Cost:** No paid API requests.



In [ ]:
compile_tables(ROOT)
print('Judgments:',len(consolidate_judgments(ROOT)))
summary = intervals(ROOT)
print('Comparison rows:',len(summary))
print('Charts input:',ROOT/'results/compiled/all_judgments.csv')